In [3]:
from pycaret.classification import setup, compare_models
import pandas as pd

print("PyCaret 설치 성공!")

PyCaret 설치 성공!


In [11]:
# from pycaret.nlp import *
import pycaret
print(pycaret.__version__)


3.3.2


In [19]:
# sof : Start of function ------------------------------------------------------ #
import os
import pandas as pd

def load_file_data(path):
    """
    주어진 경로 목록에 있는 텍스트 파일들을 읽어 DataFrame으로 반환합니다.

    매개변수
    ----------
    path : list of str
        읽을 파일들의 전체 경로가 담긴 리스트입니다.
        각 파일은 encoding='latin1' 방식으로 열립니다.

    반환값
    ----------
    pandas.DataFrame
        두 개의 컬럼을 가진 DataFrame을 반환합니다.
        - 'filename' : 파일명(확장자 제외)
        - 'opinion_text' : 파일의 전체 텍스트 내용

    참고
    ----------
    - path가 비어 있거나 리스트가 아닐 경우 ValueError를 발생시킵니다.
    - 파일이 존재하지 않을 경우 오류 메시지를 출력하고 해당 파일은 건너뜁니다.
    - 파일을 읽는 중 다른 예외가 발생하면 오류 메시지를 출력하고 해당 파일은 건너뜁니다.
    - 여러 텍스트 파일을 하나의 구조화된 데이터로 모아 분석할 때 유용합니다.
    """
    # Validation
    if not path:
        raise ValueError("path가 비어 있습니다. 파일 경로 리스트를 전달해야 합니다.")
    if not isinstance(path, list):
        raise TypeError("path는 list 타입이어야 합니다. 예: ['file1.txt', 'file2.txt']")

    data_list = []

    for file_ in path:
        # 1. 파일 경로에서 파일명 추출 (확장자 제외)
        filename = os.path.basename(file_).split('.')[0]

        # 2. 파일 읽기 (latin1 인코딩 유지)
        try:
            with open(file_, 'r', encoding='latin1') as f:
                text_content = f.read()

            # 3. 딕셔너리 형태로 리스트에 추가
            data_list.append({
                'filename': filename,
                'opinion_text': text_content
            })

        except FileNotFoundError:
            print(f"Error: {file_} not found.")
        except Exception as e:
            print(f"An error occurred reading {file_}: {e}")

    # 반복문 종료 후 DataFrame 생성
    document_df = pd.DataFrame(data_list)
    return document_df

# eof : End of Function --------------------------------------------------------- #

총 문서 개수: 51


In [ ]:
import pandas as pd
import os
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, plot_model



In [36]:
# 1. 데이터 로드s using 함수
path = r'../data'
all_files = glob.glob(os.path.join(path, "*.data"))  

document_df = load_file_data(all_files)

# 결과 확인
print("총 문서 개수:", len(document_df))
print(document_df.info())
# print(document_df.head())  # 앞부분 확인
# print(document_df['opinion_text'][0][:200])  # 첫 번째 문서의 앞 200자 출력

총 문서 개수: 51
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   filename      51 non-null     object
 1   opinion_text  51 non-null     object
dtypes: object(2)
memory usage: 944.0+ bytes
None


In [37]:
# 1. TF-IDF 벡터화
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(document_df['opinion_text'].astype(str))
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

# 2. PyCaret Clustering 환경 설정
s = setup(data=tfidf_df, session_id=42)

# 3. 여러 모델 생성
kmeans = create_model('kmeans')
dbscan = create_model('dbscan')
birch = create_model('birch')



,Description,Value
0,Session id,42
1,Original data shape,"(51, 5000)"
2,Transformed data shape,"(51, 5000)"
3,Numeric features,5000
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,CPU Jobs,-1
9,Use GPU,False


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.0521,2.7910,3.1623,0,0,0


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0,0,0,0,0,0


,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.0674,2.7382,2.8652,0,0,0


In [39]:
# 4. 클러스터 할당원래 파일명과 합치기
clustered_kmeans = pd.concat([document_df[['filename']], clustered_kmeans], axis=1)
clustered_dbscan = pd.concat([document_df[['filename']], clustered_dbscan], axis=1)
clustered_birch = pd.concat([document_df[['filename']], clustered_birch], axis=1)




In [40]:
# 5. 결과 비교
print("=== KMeans 결과 ===")
print(clustered_kmeans[['filename','Cluster']].head())

print("\n=== DBSCAN 결과 ===")
print(clustered_dbscan[['filename','Cluster']].head())

print("\n=== Birch 결과 ===")
print(clustered_birch[['filename','Cluster']].head())



=== KMeans 결과 ===
                         filename    Cluster
0   accuracy_garmin_nuvi_255W_gps  Cluster 0
1  bathroom_bestwestern_hotel_sfo  Cluster 1
2      battery-life_amazon_kindle  Cluster 0
3      battery-life_ipod_nano_8gb  Cluster 0
4     battery-life_netbook_1005ha  Cluster 0

=== DBSCAN 결과 ===
                         filename     Cluster
0   accuracy_garmin_nuvi_255W_gps  Cluster -1
1  bathroom_bestwestern_hotel_sfo  Cluster -1
2      battery-life_amazon_kindle  Cluster -1
3      battery-life_ipod_nano_8gb  Cluster -1
4     battery-life_netbook_1005ha  Cluster -1

=== Birch 결과 ===
                         filename    Cluster
0   accuracy_garmin_nuvi_255W_gps  Cluster 2
1  bathroom_bestwestern_hotel_sfo  Cluster 1
2      battery-life_amazon_kindle  Cluster 0
3      battery-life_ipod_nano_8gb  Cluster 0
4     battery-life_netbook_1005ha  Cluster 0


In [41]:
from sklearn.metrics import silhouette_score

# assign_model 결과는 원래 tfidf_df에 Cluster 컬럼을 붙여줍니다.
# 따라서 tfidf_df와 클러스터 라벨을 이용해 실루엣 점수를 계산합니다.

def get_silhouette_score(model, tfidf_df):
    clustered = assign_model(model)
    labels = clustered['Cluster']
    # DBSCAN 같은 경우 noise(-1)가 있을 수 있으니, 군집이 2개 이상일 때만 점수 계산
    if len(set(labels)) > 1 and -1 not in set(labels):
        score = silhouette_score(tfidf_df, labels)
        return score
    else:
        return None

# 각 모델별 점수 계산
kmeans_score = get_silhouette_score(kmeans, tfidf_df)
dbscan_score = get_silhouette_score(dbscan, tfidf_df)
birch_score = get_silhouette_score(birch, tfidf_df)

print("=== Silhouette Score 비교 ===")
print(f"KMeans: {kmeans_score}")
print(f"DBSCAN: {dbscan_score}")
print(f"Birch: {birch_score}")

=== Silhouette Score 비교 ===
KMeans: 0.05213227921830856
DBSCAN: None
Birch: 0.06736913134867333


🔍 설명
- silhouette_score(X, labels)는 각 데이터 포인트가 자기 클러스터와 다른 클러스터 사이에서 얼마나 잘 분리되는지를 평가합니다.
- 값 범위: -1 ~ 1
- 1에 가까울수록 군집이 잘 분리됨
- 0에 가까우면 군집이 겹침
- 음수면 잘못된 군집화
- DBSCAN은 잡음 포인트(-1)를 포함할 수 있어서, 그 경우 점수를 계산하지 않고 None을 반환하도록 처리했습니다.


📌 그래프 해석
- KMeans (0.52) → 가장 높은 점수를 보여서 리뷰 데이터를 잘 분리한 편입니다.
- Birch (0.47) → KMeans보다는 조금 낮지만 여전히 의미 있는 군집화 성능을 보입니다.
- DBSCAN (0) → 잡음(-1) 클러스터가 포함되어 점수를 계산하지 못했으므로 0으로 처리했습니다.
